In [ ]:
#!pip install crewai crewai_tools langchain langchain_community langchain_ollama streamlit duckduckgo-search

In [1]:
from com.example.agentic.agent.LLMManager import LLMManager
# groq, openai
llm = LLMManager.create_llm('ollama')
llm.call('Hello')


'Hello! How can I assist you today?'

In [ ]:
from crewai.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults 
import json


@tool
def search_web_tool(query: str):
    """
    Searches the web and returns results.
    """
    search_tool = DuckDuckGoSearchResults(num_results=5, verbose=True)
    return search_tool.run(query)


In [2]:
from crewai import Agent

# Agent Resercher
guide_expert= Agent( 
    role="City Local Guide Expert",
    goal="Provides information on things to do in the city based on the user's interests.",
    backstory="""A local expert with a passion for sharing the best experiences and hidden gems of their city.""",
    tools=[search_web_tool],
    verbose=True,
    max_iter=2,
    #llm=llm,  #ChatOpenAI(temperature=0, model="gpt-4o-mini"),
    allow_delegation=False,
)

In [3]:
# Agent city expert
location_expert = Agent(
    role="Travel Trip Expert",
    goal="Adapt to the user destination city language (French if city in French Country. Gather helpful information about to the city and city during travel.",
    backstory="""A seasoned traveler who has explored various destinations and knows the ins and outs of travel logistics.""",
    tools=[search_web_tool],
    verbose=True,
    max_iter=2,
    #llm=llm,
    allow_delegation=False,
)


In [4]:
planner_expert = Agent(
    role="Travel Planning Expert",
    goal="Compiles all gathered information to provide a comprehensive travel plan.",
    backstory="""
    You are a professional guide with a passion for travel.
    An organizational wizard who can turn a list of possibilities into a seamless itinerary.
    """,
    tools=[search_web_tool],
    verbose=True,
    max_iter=2,
    #llm=llm,
    allow_delegation=False,
)


In [5]:
from datetime import datetime
from crewai import Task

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

from_city = "India"
destination_city = "Rome"
date_from = "1st March 2025"
date_to = "7th March 2025"
interests = "sight seeing and good food"

location_task = Task(
    description=f"""
    In French : This task involves a comprehensive data collection process to provide the traveler with essential information about their destination. It includes researching and compiling details on various accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of living in the area. The task also covers transportation options, visa requirements, and any travel advisories that may be relevant.
    consider also the weather conditions forcast on the travel dates. and all the events that may be relevant to the traveler during the trip period.
    
    Traveling from : {from_city}
    Destination city : {destination_city}
    Arrival Date : {date_from}
    Departure Date : {date_to}

    Follow this rules : 
    1. if the {destination_city} is in a French country : Respond in FRENCH.
    """,
    expected_output=f"""
    if the {destination_city} is in a French country : Respond in FRENCH.
    In markdown format : A detailed markdown report that includes a curated list of recommended places to stay, a breakdown of daily living expenses, and practical travel tips to ensure a smooth journey.
    """,
    agent=location_expert,
    output_file=f'outputs/L002/city_report_{run_id}.md',
)





In [6]:
guide_task = Task(
    description=f"""
    if the {destination_city} is in a French country : Respond in FRENCH.
    Tailored to the traveler's personal {interests}, this task focuses on creating an engaging and informative guide to the city's attractions. It involves identifying cultural landmarks, historical spots, entertainment venues, dining experiences, and outdoor activities that align with the user's preferences such {interests}. The guide also highlights seasonal events and festivals that might be of interest during the traveler's visit.
    Destination city : {destination_city}
    interests : {interests}
    Arrival Date : {date_from}
    Departure Date : {date_to}

    Follow this rules : 
    1. if the {destination_city} is in a French country : Respond in FRENCH.
    """,
    expected_output=f"""
    An interactive markdown report that presents a personalized itinerary of activities and attractions, complete with descriptions, locations, and any necessary reservations or tickets.
    """,

    agent=guide_expert,
    output_file=f'outputs/L002/guide_report_{run_id}.md',
)


In [7]:
planner_task = Task(
    description=f"""
    This task synthesizes all collected information into a detaileds introduction to the city (description of city and presentation, in 3 pragraphes) cohesive and practical travel plan. and takes into account the traveler's schedule, preferences, and budget to draft a day-by-day itinerary. The planner also provides insights into the city's layout and transportation system to facilitate easy navigation.
    Destination city : {destination_city}
    interests : {interests}
    Arrival Date : {date_from}
    Departure Date : {date_to}

    Follow this rules : 
    1. if the {destination_city} is in a French country : Respond in FRENCH.
    """,
    expected_output="""
    if the {destination_city} is in a French country : Respond in FRENCH.
    A rich markdown document with emojis on each title and subtitle, that :
    In markdown format : 
    # Welcome to {destination_city} :
    A 4 paragraphes markdown formated including :
    - a curated articles of presentation of the city, 
    - a breakdown of daily living expenses, and spots to visit.
    # Here's your Travel Plan to {destination_city} :
    Outlines a daily detailed travel plan list with time allocations and details for each activity, along with an overview of the city's highlights based on the guide's recommendations
    """,
    context=[location_task, guide_task],
    #context=context,
    agent=planner_expert,
    output_file=f'outputs/L002/travel_plan_{run_id}.md',
)


In [8]:
# Task: Location
def location_task(agent, from_city, destination_city, date_from, date_to):
    return Task(
        description=f"""
        In French : This task involves a comprehensive data collection process to provide the traveler with essential information about their destination. It includes researching and compiling details on various accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of living in the area. The task also covers transportation options, visa requirements, and any travel advisories that may be relevant.
        consider also the weather conditions forcast on the travel dates. and all the events that may be relevant to the traveler during the trip period.
        
        Traveling from : {from_city}
        Destination city : {destination_city}
        Arrival Date : {date_from}
        Departure Date : {date_to}

        Follow this rules : 
        1. if the {destination_city} is in a French country : Respond in FRENCH.
        """,
        expected_output=f"""
        if the {destination_city} is in a French country : Respond in FRENCH.
        In markdown format : A detailed markdown report that includes a curated list of recommended places to stay, a breakdown of daily living expenses, and practical travel tips to ensure a smooth journey.
        """,
        agent=agent,
        output_file=f'outputs/L002/city_report_{run_id}.md',
    )

# Task: Location
def guide_task(agent, destination_city, interests, date_from, date_to):    
    return Task(
        description=f"""
        if the {destination_city} is in a French country : Respond in FRENCH.
        Tailored to the traveler's personal {interests}, this task focuses on creating an engaging and informative guide to the city's attractions. It involves identifying cultural landmarks, historical spots, entertainment venues, dining experiences, and outdoor activities that align with the user's preferences such {interests}. The guide also highlights seasonal events and festivals that might be of interest during the traveler's visit.
        Destination city : {destination_city}
        interests : {interests}
        Arrival Date : {date_from}
        Departure Date : {date_to}

        Follow this rules : 
        1. if the {destination_city} is in a French country : Respond in FRENCH.
        """,
        expected_output=f"""
        An interactive markdown report that presents a personalized itinerary of activities and attractions, complete with descriptions, locations, and any necessary reservations or tickets.
        """,

        agent=agent,
        output_file=f'outputs/L002/guide_report_{run_id}.md',
    )


# Task: Planner
def planner_task(context, agent, destination_city, interests, date_from, date_to):
    return Task(
        description=f"""
        This task synthesizes all collected information into a detaileds introduction to the city (description of city and presentation, in 3 pragraphes) cohesive and practical travel plan. and takes into account the traveler's schedule, preferences, and budget to draft a day-by-day itinerary. The planner also provides insights into the city's layout and transportation system to facilitate easy navigation.
        Destination city : {destination_city}
        interests : {interests}
        Arrival Date : {date_from}
        Departure Date : {date_to}

        Follow this rules : 
        1. if the {destination_city} is in a French country : Respond in FRENCH.
        """,
        expected_output="""
        if the {destination_city} is in a French country : Respond in FRENCH.
        A rich markdown document with emojis on each title and subtitle, that :
        In markdown format : 
        # Welcome to {destination_city} :
        A 4 paragraphes markdown formated including :
        - a curated articles of presentation of the city, 
        - a breakdown of daily living expenses, and spots to visit.
        # Here's your Travel Plan to {destination_city} :
        Outlines a daily detailed travel plan list with time allocations and details for each activity, along with an overview of the city's highlights based on the guide's recommendations
        """,
        #context=[location_task, guide_task],
        context=context,
        agent=agent,
        output_file=f'outputs/L002/travel_plan_{run_id}.md',
    )


In [9]:

location_task = location_task(
  location_expert,
  from_city,
  destination_city,
  date_from,
  date_to
)

guide_task = guide_task(
  guide_expert,
  destination_city,
  interests,
  date_from,
  date_to
)

planner_task = planner_task(
  [location_task, guide_task],
  planner_expert,
  destination_city,
  interests,
  date_from,
  date_to,
)



In [10]:
from crewai import Crew, Process
crew = Crew(
    agents=[location_expert, guide_expert, planner_expert],
    tasks=[location_task, guide_task, planner_task],
    process=Process.sequential,
    full_output=True,
    share_crew=False,
    verbose=True
)

result = crew.kickoff()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 999b0ff4-0180-4dd9-b1f8-30f1ac1dc085                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          In French : This task involves a comprehensive data collection process to provide the traveler with    │
│  essential information about their destination. It includes researching and compiling details on various        │
│  accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of       │
│  living in the area. The task also covers transportation options, visa requirements, and any travel advisories  │
│  that may be relevant.                                                                                          │
│          consider also the weather conditions forcast on the travel dates. and all the events that may be       │
│  relevant to the traveler during the trip period.                                                               │
│                                                                                                                 │
│          Traveling from : India                                                                                 │
│          Destination city : Rome                                                                                │
│          Arrival Date : 1st March 2025                                                                          │
│          Departure Date : 7th March 2025                                                                        │
│                                                                                                                 │
│          Follow this rules :                                                                                    │
│          1. if the Rome is in a French country : Respond in FRENCH.                                             │
│                                                                                                                 │
│  ID: ee9ce25f-16a0-41b1-ae45-9b69352930d3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Trip Expert                                                                                      │
│                                                                                                                 │
│  Task:                                                                                                          │
│          In French : This task involves a comprehensive data collection process to provide the traveler with    │
│  essential information about their destination. It includes researching and compiling details on various        │
│  accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of       │
│  living in the area. The task also covers transportation options, visa requirements, and any travel advisories  │
│  that may be relevant.                                                                                          │
│          consider also the weather conditions forcast on the travel dates. and all the events that may be       │
│  relevant to the traveler during the trip period.                                                               │
│                                                                                                                 │
│          Traveling from : India                                                                                 │
│          Destination city : Rome                                                                                │
│          Arrival Date : 1st March 2025                                                                          │
│          Departure Date : 7th March 2025                                                                        │
│                                                                                                                 │
│          Follow this rules :                                                                                    │
│          1. if the Rome is in a French country : Respond in FRENCH.                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Trip Expert                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Comprehensive Travel Report for Rome (Italy)                                                                │
│                                                                                                                 │
│  ### Parameters                                                                                                 │
│  ```                                                                                                            │
│  {                                                                                                              │
│    "destination": {                                                                                             │
│      "name": "Rome",                                                                                            │
│      "country": "France"                                                                                        │
│    },                                                                                                           │
│    "country_name": "France",                                                                                    │
│    "departure_date": "1st March 2025",                                                                          │
│    "arrival_date": "7th March 2025"                                                                             │
│  }                                                                                                              │
│  ```                                                                                                            │
│                                                                                                                 │
│  ### Response                                                                                                   │
│  ```json                                                                                                        │
│  {                                                                                                              │
│    "name": "Travel Trip Expert",                                                                                │
│    "parameters": {                                                                                              │
│      "destination": {                                                                                           │
│        "name": "Rome",                                                                                          │
│        "country": "France"                                                                                      │
│      },                                                                                                         │
│      "country_name": "France",                                                                                  │
│      "departure_date": "1st March 2025",                                                                        │
│      "arrival_date": "7th March 2025"                                                                           │
│    }                                                                                                            │
│  }                                                                                                              │
│  ```                                                   

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│          In French : This task involves a comprehensive data collection process to provide the traveler with    │
│  essential information about their destination. It includes researching and compiling details on various        │
│  accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of       │
│  living in the area. The task also covers transportation options, visa requirements, and any travel advisories  │
│  that may be relevant.                                                                                          │
│          consider also the weather conditions forcast on the travel dates. and all the events that may be       │
│  relevant to the traveler during the trip period.                                                               │
│                                                                                                                 │
│          Traveling from : India                                                                                 │
│          Destination city : Rome                                                                                │
│          Arrival Date : 1st March 2025                                                                          │
│          Departure Date : 7th March 2025                                                                        │
│                                                                                                                 │
│          Follow this rules :                                                                                    │
│          1. if the Rome is in a French country : Respond in FRENCH.                                             │
│                                                                                                                 │
│  Agent: Travel Trip Expert                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Planning Expert                                                                                  │
│                                                                                                                 │
│  Task:                                                                                                          │
│          This task synthesizes all collected information into a detaileds introduction to the city              │
│  (description of city and presentation, in 3 pragraphes) cohesive and practical travel plan. and takes into     │
│  account the traveler's schedule, preferences, and budget to draft a day-by-day itinerary. The planner also     │
│  provides insights into the city's layout and transportation system to facilitate easy navigation.              │
│          Destination city : Rome                                                                                │
│          interests : sight seeing and good food                                                                 │
│          Arrival Date : 1st March 2025                                                                          │
│          Departure Date : 7th March 2025                                                                        │
│                                                                                                                 │
│          Follow this rules :                                                                                    │
│          1. if the Rome is in a French country : Respond in FRENCH.                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Planning Expert                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│    "name": "Travel Trip Expert",                                                                                │
│    "parameters": {                                                                                              │
│      "destination": {                                                                                           │
│        "name": "Rome",                                                                                          │
│        "country": "France"                                                                                      │
│      },                                                                                                         │
│      "country_name": "France",                                                                                  │
│      "departure_date": "1st March 2025",                                                                        │
│      "arrival_date": "7th March 2025"                                                                           │
│    }                                                                                                            │
│  }                                                                                                              │
│  ```                                                                                                            │
│                                                                                                                 │
│  **L'expérience de voyage pour Rome (Italie)**                                                                  │
│                                                                                                                 │
│  **Informations communes**                                                                                      │
│                                                                                                                 │
│  * La période d'arrivée est la première du 1er mars 2025 au 7ème mars 2025.                                     │
│  * Pour des informations supplémentaires, veuillez contacter notre équipe à votre plus proche bureau.           │
│                                                                                                                 │
│  **Accommodations**                                                                                             │
│                                                                                                                 │
│  * Le Palazzo Quarto                                                                                            │
│          + Recommandation : Hotel Le Palazzo Quarto                                                             │
│      - Résumé de la localisation de l'hôtel (exposition, voie, quartier)                                        │
│          Présentation du hôtel (description des chambres, services offerts)                                     │
│  * Enoteca della Lombarda                                                                                       │
│          + Recommandation : Hotel Enoteca della Lombard

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│          This task synthesizes all collected information into a detaileds introduction to the city              │
│  (description of city and presentation, in 3 pragraphes) cohesive and practical travel plan. and takes into     │
│  account the traveler's schedule, preferences, and budget to draft a day-by-day itinerary. The planner also     │
│  provides insights into the city's layout and transportation system to facilitate easy navigation.              │
│          Destination city : Rome                                                                                │
│          interests : sight seeing and good food                                                                 │
│          Arrival Date : 1st March 2025                                                                          │
│          Departure Date : 7th March 2025                                                                        │
│                                                                                                                 │
│          Follow this rules :                                                                                    │
│          1. if the Rome is in a French country : Respond in FRENCH.                                             │
│                                                                                                                 │
│  Agent: Travel Planning Expert                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 999b0ff4-0180-4dd9-b1f8-30f1ac1dc085                                                                       │
│  Final Output: ```json                                                                                          │
│  {                                                                                                              │
│    "name": "Travel Trip Expert",                                                                                │
│    "parameters": {                                                                                              │
│      "destination": {                                                                                           │
│        "name": "Rome",                                                                                          │
│        "country": "France"                                                                                      │
│      },                                                                                                         │
│      "country_name": "France",                                                                                  │
│      "departure_date": "1st March 2025",                                                                        │
│      "arrival_date": "7th March 2025"                                                                           │
│    }                                                                                                            │
│  }                                                                                                              │
│  ```                                                                                                            │
│                                                                                                                 │
│  **L'expérience de voyage pour Rome (Italie)**                                                                  │
│                                                                                                                 │
│  **Informations communes**                                                                                      │
│                                                                                                                 │
│  * La période d'arrivée est la première du 1er mars 2025 au 7ème mars 2025.                                     │
│  * Pour des informations supplémentaires, veuillez contacter notre équipe à votre plus proche bureau.           │
│                                                                                                                 │
│  **Accommodations**                                                                                             │
│                                                                                                                 │
│  * Le Palazzo Quarto                                                                                            │
│          + Recommandation : Hotel Le Palazzo Quarto                                                             │
│      - Résumé de la localisation de l'hôtel (exposition, voie, quartier)                                        │
│          Présentation du hôtel (description des chambres, services offerts)                                     │
│  * Enoteca della Lombarda                                                                                       │
│          + Recommandation : Hotel Enoteca della Lombar

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 32e861d6-f7d6-48be-ac86-474805a78e8b                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/32e861d6-f7d6-48b │
│ e-ac86-474805a78e8b?access_code=TRACE-523fce69b3                             │
│ 🔑 Access Code: TRACE-523fce69b3                                             │
╰──────────────────────────────────────────────────────────────────────────────╯
